In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from datasets import load_dataset
import re

2025-07-12 09:42:59.458168: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-12 09:42:59.611393: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752293579.705110    3352 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752293579.734441    3352 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752293579.884906    3352 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Josephgflowers/Finance-Instruct-500k")

In [3]:
train_ds = ds["train"]
train_ds=train_ds.to_pandas()

In [4]:
train_ds.head(1)
print(train_ds.shape)

(518185, 3)


In [6]:
assistant= train_ds['assistant'].to_list()[:1000]
user= train_ds['user'].to_list()[:1000]

In [7]:
tokenizer= tf.keras.preprocessing.text.Tokenizer()

In [8]:
tokenizer.fit_on_texts(assistant+user)
total_words = len(tokenizer.word_index) + 1

In [9]:
print(total_words)

8144


In [10]:
max_len = max([len(x) for x in user])

In [11]:
max_len

1795

In [13]:
input_sequences = []
for sentence in user:
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

  for i in range(1,len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])

In [14]:
output_sequences = []
for sentence in assistant:
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

  for i in range(1,len(tokenized_sentence)):
    output_sequences.append(tokenized_sentence[:i+1])

In [11]:
paired_sequences = []
for q, a in zip(user[:100], assistant[:100]):
    # Tokenize question and answer
    q_tokens = tokenizer.texts_to_sequences([q])[0]
    a_tokens = tokenizer.texts_to_sequences([a])[0]
    
    # Limit length
    if len(q_tokens) > 50:
        q_tokens = q_tokens[:50]
    if len(a_tokens) > 50:
        a_tokens = a_tokens[:50]
    
    # Create input (question) and target (answer) pairs
    paired_sequences.append((q_tokens, a_tokens))

In [12]:
inputs = [pair[0] for pair in paired_sequences]
targets = [pair[1] for pair in paired_sequences]

In [15]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen = max_len, padding='pre')
padded_output_sequences = pad_sequences(output_sequences, maxlen = max_len, padding='pre')

In [13]:
# Pad sequences
padded_inputs = pad_sequences(inputs, maxlen=50, padding='pre')
padded_targets = pad_sequences(targets, maxlen=50, padding='pre')

In [17]:
EMBEDDING_DIM = 64 
VOCAB_SIZE = len(tokenizer.word_index) + 1
model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_INPUT_LENGTH),
    LSTM(64, dropout=0.2),  # Single LSTM layer
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(VOCAB_SIZE, activation='softmax')
])

NameError: name 'MAX_INPUT_LENGTH' is not defined